# Compiler vs ARM-FM baseline on MiniGrid DoorKey

Generate the four `examples/arm-fm/minigrid-doorkey/tasks.json` tasks with both Reward
Machine generators, twice each and independently, render every Reward Machine to SVG, and
keep one run log per run.

- **ARM-FM baseline:** direct paper-format generation (`rm_generator`) refined by the
  packaged ARM-FM `rm_critic` for at most three attempts; no clause restrictions.
- **Compiler:** proposal -> task critic -> DECLARE/LTLf -> FL-AT/MONA DFA -> RM -> RM
  critic, bounded to `Existence`, `ExistenceTwo`, and `Precedence`.
- **Provider and model:** `.env` as-is; no provider or model override.

The comparison is semantic. The baseline serializes `u0...un` paper RMs while the compiler
serializes the numeric `s/i/f/r` format, so text is never compared byte-for-byte
(`docs/decisions/arm-fm/reproduction-boundary.md`).

Artifacts land under `docs/artifacts/arm-fm-vs-compiler/run-N/<pipeline>/` as `task-N.rm`
plus `task-N.svg`, with one `run-log.json` per run. Existing files are never overwritten:
move a run directory aside before executing that run again.

**This notebook calls the configured provider.** Execute it from the repository root.


In [1]:
import json
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path

import dotenv

from scripts.generate_rm import generate_rm
from scripts.render_rm import paper_to_structure, render_structure_svg
from src.arm_fm import serialize_paper_reward_machine
from src.arm_fm.generation import StageAttempt, generate_reward_machine
from src.config import Configuration
from src.models import EnvironmentDescription
from src.utils import GenerationHooks, ProgressEvent, StepState, get_engine

dotenv.load_dotenv()


True

## Settings

`CONFIG` reads `.env`. `EXPERIMENT` keeps the experiment-only settings: the task manifest,
its declared environment file, both runs, both pipelines, the artifact directory, the
critic toggles, the critic-free fallback, and the baseline attempt bound.


In [2]:
CONFIG = Configuration()

EXAMPLES = CONFIG.WORKSPACE_PATH / "examples" / "arm-fm" / "minigrid-doorkey"

EXPERIMENT = {
    "tasks_file": EXAMPLES / "tasks.json",
    "environment_file": EXAMPLES / "environment.md",
    "runs": (1, 2, 3),
    "pipelines": ("arm-fm", "compiler"),
    "artifact_directory": CONFIG.WORKSPACE_PATH / "docs" / "artifacts" / "arm-fm-vs-compiler",
    "critics": {"task_critic": True, "rm_critic": True},
    "critics_off_fallback": {"task_critic": False, "rm_critic": False},
    "max_attempts": 3,
}

MANIFEST = json.loads(EXPERIMENT["tasks_file"].read_text(encoding="utf-8"))
DECLARED_ENVIRONMENT = CONFIG.WORKSPACE_PATH / MANIFEST["env_description"]
if DECLARED_ENVIRONMENT != EXPERIMENT["environment_file"]:
    raise ValueError(f"{EXPERIMENT['tasks_file']} declares {DECLARED_ENVIRONMENT}")

TASKS = tuple(
    (f"task-{index}", task) for index, task in enumerate(MANIFEST["tasks"], start=1)
)
ENVIRONMENT = EnvironmentDescription.from_file(EXPERIMENT["environment_file"])

print(f"Provider '{CONFIG.llm_provider}' with model '{CONFIG.model}'.")
print(f"{len(TASKS)} tasks x {len(EXPERIMENT['runs'])} runs x {len(EXPERIMENT['pipelines'])} pipelines.")


Provider 'opencode' with model 'mimo-v2.6-flash'.
4 tasks x 3 runs x 2 pipelines.


## Helpers

Every run-log entry comes from `GenerationHooks.event` (`ProgressEvent`) for the compiler
and from the baseline's own stage attempts for ARM-FM; console text is never scraped. A
record's `status` is one of:

- `accepted`: the pipeline accepted a Reward Machine.
- `rejected`: the baseline kept its last parseable candidate after its own `rm_critic`
  never accepted (`rm_critic: rejected (fallback)`).
- `refused`: a deterministic unsupported template or undeclared proposition; never retried
  with critics off because the critics are not the cause.
- `failed`: no artifact could be produced; `reason` holds the verifier's own detail.


In [3]:
REFUSAL_MARKERS = ("unsupported declare pattern", "invented proposition")


def log_entries(attempts):
    """Return ordered run-log entries from (attempt, stage, state, detail) tuples."""
    return [
        {"attempt": attempt, "stage": stage, "state": state, "detail": detail or ""}
        for attempt, stage, state, detail in attempts
    ]


def compiler_entries(events):
    """Return run-log entries for the compiler's own stage events."""
    return log_entries(
        (event.attempt, event.step.value, event.state.value, event.detail)
        for event in events
    )


def arm_fm_entries(attempts):
    """Return run-log entries for the baseline's own stage attempts."""
    return log_entries(
        (item.attempt, item.stage, item.status, item.feedback or item.error)
        for item in attempts
    )


def refusal_detail(events):
    """Return a deterministic refusal detail, if the events show one."""
    refusals = [
        event.detail for event in events
        if event.state is StepState.FAILED and any(
            marker in (event.detail or "").lower() for marker in REFUSAL_MARKERS
        )
    ]
    return refusals[-1] if refusals else ""


def failure_detail(events):
    """Return the last failed stage detail, which is the verifier's own reason."""
    details = [
        event.detail for event in events
        if event.state is StepState.FAILED and event.detail
    ]
    return details[-1] if details else ""


def record(run, pipeline, task_id, task, status, entries, *, fallback="", reason="", artifacts=()):
    """Assemble one run-log record for one task and pipeline."""
    return {
        "run": run,
        "pipeline": pipeline,
        "task_id": task_id,
        "task": task,
        "status": status,
        "fallback": fallback,
        "reason": reason,
        "attempts": max((entry["attempt"] for entry in entries), default=0),
        "events": entries,
        "artifacts": list(artifacts),
    }


def write_artifact(path, text):
    """Write one artifact, refusing to replace an earlier run's file."""
    if path.exists():
        raise RuntimeError(f"Refusing to overwrite {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def fresh_directory(path):
    """Create one artifact directory, refusing to reuse one that holds files."""
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"{path} already holds artifacts; move it aside first")
    path.mkdir(parents=True, exist_ok=True)
    return path


In [4]:
def compile_task(task, task_id, critics):
    """Run the compiler once for one task and capture its stage events in memory."""
    events = []
    accepted = []
    hooks = GenerationHooks(
        event=events.append,
        completion=lambda results, _paths: accepted.extend(results),
    )
    task_config = replace(
        CONFIG,
        environment=EXPERIMENT["environment_file"],
        tasks=[task],
        output=Path(f"{task_id}.rm"),
        overwrite=True,
        **critics,
    )
    status = generate_rm(task_config, hooks, write_outputs=False)
    return status, events, accepted


def compiler_record(run, task_id, task, directory):
    """Compile one task, falling back to a critic-free rerun unless it was refused."""
    status, events, accepted = compile_task(task, task_id, EXPERIMENT["critics"])
    refusal = refusal_detail(events)

    if status != 0 and refusal:
        return record(
            run, "compiler", task_id, task, "refused",
            compiler_entries(events), reason=refusal,
        )

    fallback = ""
    if status != 0:
        fallback = "critics off (fallback)"
        status, events, accepted = compile_task(task, task_id, EXPERIMENT["critics_off_fallback"])

    entries = compiler_entries(events)
    if status != 0 or not accepted:
        return record(
            run, "compiler", task_id, task, "failed", entries,
            fallback=fallback,
            reason=failure_detail(events) or "compiler returned no accepted result",
        )

    result = accepted[0]
    write_artifact(directory / f"{task_id}.rm", result.text)
    write_artifact(directory / f"{task_id}.svg", render_structure_svg(result.reward_machine))
    return record(
        run, "compiler", task_id, task, "accepted", entries,
        fallback=fallback, artifacts=(f"{task_id}.rm", f"{task_id}.svg"),
    )


def arm_fm_record(run, task_id, task, directory, engine):
    """Generate one baseline RM, keeping the last parseable candidate as a fallback."""
    machine, _text, attempts = generate_reward_machine(
        task, ENVIRONMENT, engine, max_attempts=EXPERIMENT["max_attempts"],
    )
    entries = arm_fm_entries(attempts)

    if machine is None:
        errors = [item.error for item in attempts if item.error]
        return record(
            run, "arm-fm", task_id, task, "failed", entries,
            reason=errors[-1] if errors else "no parseable Reward Machine",
        )

    try:
        svg = render_structure_svg(paper_to_structure(machine))
    except ValueError as error:
        return record(
            run, "arm-fm", task_id, task, "failed", entries,
            reason=f"graph rendering failed: {error}",
        )

    accepted = any(item.status == "accepted" for item in attempts)
    write_artifact(directory / f"{task_id}.rm", serialize_paper_reward_machine(machine))
    write_artifact(directory / f"{task_id}.svg", svg)
    return record(
        run, "arm-fm", task_id, task, "accepted" if accepted else "rejected", entries,
        fallback="" if accepted else "rm_critic: rejected (fallback)",
        artifacts=(f"{task_id}.rm", f"{task_id}.svg"),
    )


## Run both pipelines

Run 1 and run 2 are independent: each pass starts with a fresh engine and per-task history.
The compiler is tried with both critics first, and a task that still exhausts its three
attempts is re-run alone with `task_critic=False, rm_critic=False` and tagged
`critics off (fallback)`. The baseline uses its own `rm_critic` loop and keeps the last
parseable candidate when that loop never accepts, tagged
`rm_critic: rejected (fallback)`.

The next cell performs real provider calls. It writes both runs' artifacts and run logs.


In [5]:
for run in EXPERIMENT["runs"]:
    run_directory = EXPERIMENT["artifact_directory"] / f"run-{run}"
    if (run_directory / "run-log.json").exists():
        print(f"run {run}: already complete, skipping -> {run_directory}")
        continue
    run_directory = fresh_directory(run_directory)
    records = []

    for pipeline in EXPERIMENT["pipelines"]:
        engine = get_engine(CONFIG, ENVIRONMENT) if pipeline == "arm-fm" else None
        directory = fresh_directory(run_directory / pipeline)

        for task_id, task in TASKS:
            print(f"run {run} | {pipeline} | {task_id}: {task}")

            if pipeline == "arm-fm":
                entry = arm_fm_record(run, task_id, task, directory, engine)
            else:
                entry = compiler_record(run, task_id, task, directory)

            records.append(entry)
            print(f"  -> {entry['status']} {entry['fallback']} {entry['reason']}".rstrip())

    accepted = [entry for entry in records if entry["status"] == "accepted"]
    run_log = {
        "run": run,
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "provider": CONFIG.llm_provider,
        "model": CONFIG.model,
        "tasks_file": str(EXPERIMENT["tasks_file"].relative_to(CONFIG.WORKSPACE_PATH)),
        "environment_file": str(
            EXPERIMENT["environment_file"].relative_to(CONFIG.WORKSPACE_PATH)
        ),
        "critics": EXPERIMENT["critics"],
        "critics_off_fallback": EXPERIMENT["critics_off_fallback"],
        "max_attempts": EXPERIMENT["max_attempts"],
        "records": records,
    }
    write_artifact(run_directory / "run-log.json", json.dumps(run_log, indent=2) + "\n")
    print(f"run {run}: {len(accepted)}/{len(records)} accepted -> {run_directory}")


run 1: already complete, skipping -> /home/turbotowerlnx/Documents/Master/TFM/TFM-schema-rm-rl/docs/artifacts/arm-fm-vs-compiler/run-1
run 2: already complete, skipping -> /home/turbotowerlnx/Documents/Master/TFM/TFM-schema-rm-rl/docs/artifacts/arm-fm-vs-compiler/run-2
run 3 | arm-fm | task-1: Use the key to open the door, then reach the goal.


  -> accepted
run 3 | arm-fm | task-2: Eventually acquire the key.


  -> accepted
run 3 | arm-fm | task-3: Open the door only after acquiring the key.


  -> accepted
run 3 | arm-fm | task-4: Reach the goal only after opening the door.


  -> accepted
run 3 | compiler | task-1: Use the key to open the door, then reach the goal.
[total 0.000s | step 0.000s] Run log: /home/turbotowerlnx/Documents/Master/TFM/TFM-schema-rm-rl/logs/run-20260922T200054.187883Z.log


...

[total 8.177s | step 0.000s] Task 1: attempt 1/3 accepted.


[total 8.177s | step 0.000s] Writing Reward Machine outputs.


[total 8.177s | step 0.000s] Writing outputs complete.


[total 8.177s | step 0.000s] Generation completed successfully.


  -> accepted critics off (fallback)
run 3: 8/8 accepted -> /home/turbotowerlnx/Documents/Master/TFM/TFM-schema-rm-rl/docs/artifacts/arm-fm-vs-compiler/run-3
